In [45]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import copy
import random
import warnings
warnings.filterwarnings('ignore')

# random seeds
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

In [46]:
# MLP config
class Config_MLP:
    use_bic: bool = False
    # Network architecture
    hidden_layers = [128, 64, 32]

    # Training parameters
    learning_rate = 0.001
    batch_size = 1024
    epochs = 100
    early_stopping_rounds = 10
    n_ensembles = 10

    # Data parameters
    initial_train_years = 3
    validation_years = 1
    test_months = 1

    # Lag candidates
    lag_candidates = [1, 5, 22]

    # Transformation parameters
    use_log_transform = True  # model log(RV) instead of rv directly
    target_scaler = True  # scale target variable

    # Device
    #if torch.backends.mps.is_available(): #mps sucks...
        #device = torch.device("mps")
       # print("Using MPS")
    #if torch.cuda.is_available(): #if you want to use cuda
        #device = torch.device('cuda')
        #print("Using CUDA")
    #else:
    device = torch.device('cpu') #default to cpu
    print("Using CPU")


Using CPU


In [47]:
# LSTM Config
class Config_LSTM:
    use_bic: bool = False
    # Network architecture
    hidden_layers = [32, 16]

    # Training parameters
    learning_rate = 0.001
    batch_size = 1024
    epochs = 100
    early_stopping_rounds = 10
    n_ensembles = 10

    # Data parameters
    initial_train_years = 3
    validation_years = 1
    test_months = 1

    # Lag candidates
    lag_candidates = [1, 5, 22]

    # Transformation parameters
    use_log_transform = True  # model log(RV) instead of rv directly
    target_scaler = True  # scale target variable

    # Device
    if torch.cuda.is_available(): #default to cuda for lstm
        device = torch.device('cuda')
        print("Using CUDA")
    elif torch.backends.mps.is_available(): #haven't tried mps on lstm
        device = torch.device("mps")
        print("Using MPS")
    else:
        device = torch.device('cpu')
        print("Using CPU")


Using MPS


In [48]:
class MLPWithBatchNorm(nn.Module):
    def __init__(self, input_dim, hidden_layers):
        super(MLPWithBatchNorm, self).__init__()

        layers = []
        prev_dim = input_dim

        # Build hidden layers with batch normalization
        for hidden_dim in hidden_layers:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.2))
            prev_dim = hidden_dim

        # Output layer - NO activation
        layers.append(nn.Linear(prev_dim, 1))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [49]:
class LSTMWithBatchNorm(nn.Module):
    def __init__(self, input_dim: int, hidden_layers, dropout: float = 0.0):
        super().__init__()

        self.input_dim = int(input_dim)  # number of lags
        self.hidden_layers = list(hidden_layers)
        self.dropout = float(dropout)

        # Build LSTM stack
        self.lstm_layers = nn.ModuleList()
        # First LSTM layer: input_size=1 (lag value), hidden_size=hidden_layers[0]
        self.lstm_layers.append(nn.LSTM(
            input_size=1,
            hidden_size=self.hidden_layers[0],
            num_layers=1,
            batch_first=True
        ))

        # Additional LSTM layers if provided
        for in_size, out_size in zip(self.hidden_layers[:-1], self.hidden_layers[1:]):
            self.lstm_layers.append(nn.LSTM(
                input_size=in_size,
                hidden_size=out_size,
                num_layers=1,
                batch_first=True
            ))

        # Optional dropout
        self.use_dropout = self.dropout > 0.0
        if self.use_dropout:
            self.dropout_layer = nn.Dropout(self.dropout)

        # BatchNorm over the final hidden size
        last_hidden = self.hidden_layers[-1]
        self.bn = nn.BatchNorm1d(last_hidden)

        # Final linear to 1-d output
        self.fc_out = nn.Linear(last_hidden, 1)

        # Initialize the output layer (optional but helps stability)
        nn.init.kaiming_uniform_(self.fc_out.weight, a=np.sqrt(5))
        if self.fc_out.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.fc_out.weight)
            bound = 1 / np.sqrt(fan_in)
            nn.init.uniform_(self.fc_out.bias, -bound, bound)

    def forward(self, x):
        if x.dim() == 2:
            # (N, num_lags) -> (N, seq_len=num_lags, 1)
            x = x.unsqueeze(-1)
        elif x.dim() == 3:
            # already (batch, seq_len, input_size)
            pass
        else:
            raise ValueError(f"Expected input of shape (N, L) or (N, L, C), got {tuple(x.shape)}")

        # Pass through LSTM stack
        for lstm in self.lstm_layers:
            x, _ = lstm(x)  # x: (batch, seq_len, hidden_size)

        # Last time step
        last = x[:, -1, :]  # (batch, hidden_size)

        # BatchNorm + optional dropout
        last = self.bn(last)
        if self.use_dropout:
            last = self.dropout_layer(last)

        out = self.fc_out(last)  # (batch, 1)
        return out


In [50]:
# Data parsing
def load_and_prepare_data(filepath):
    rv = pd.read_csv(filepath)
    rv = rv[["Date", "Volatility", "Type"]]
    rv.rename(columns={"Volatility": "RV_daily"}, inplace=True)
    rv = rv[rv['Type'] == 'QMLE-Trade']
    rv.drop(columns=['Type'], inplace=True)
    rv = rv.set_index("Date")
    rv.index = pd.to_datetime(rv.index)
    rv.index.name = "date"

    print(f"Data loaded: {len(rv)} observations")
    print(f"Date range: {rv.index.min()} to {rv.index.max()}")

    return rv

def load_multiple_tickers(tickers, data_dir="."):
    out = {}
    for t in tickers:
        fp = f"{data_dir}/{t}.csv"
        print(f"Loading {t} from {fp} ...")
        rv = load_and_prepare_data(fp)
        out[t] = rv
    return out

def prepare_train_val_test_data(df, train_start, train_end, val_end, test_end): #split into train, val, test
    train_data = df[train_start:train_end]
    val_data = df[train_end:val_end]
    test_data = df[val_end:test_end]

    return train_data, val_data, test_data

def create_lagged_features_multi(rv_dict, lags):
    frames = []
    for t, df in rv_dict.items():
        lagged = create_lagged_features(df.copy(), lags)
        lagged['ticker'] = t
        frames.append(lagged)
    pooled = pd.concat(frames, axis=0).sort_index()
    return pooled

def create_lagged_features(data, lags):
    df = data.copy()

    # Create lagged features
    for lag in lags:
        df[f'lag_{lag}'] = df['RV_daily'].shift(lag)

    # Remove rows with NaN values (from lagging)
    df = df.dropna()

    return df

In [51]:
# Calculate BIC
from statsmodels.tsa.ar_model import AutoReg

def calculate_BIC(rv_data, max_lag: int = 44):
    bic_values = {}
    for lag in range(2, max_lag + 1):
        model = AutoReg(rv_data.iloc[-252:], lags=lag, old_names=False)
        result = model.fit()
        bic_values[lag] = result.bic

    best_lag = min(bic_values, key=bic_values.get)

    chosen = [1]

    if best_lag > 10:
        second_var = int(best_lag * 5/22)
        chosen.append(second_var)
        chosen.append(best_lag)
    else:
        second_var = int(best_lag * 22/5)
        chosen.append(best_lag)
        chosen.append(second_var)

    # keep sorted by size for stable column ordering
    chosen = sorted(chosen)
    print(f"BIC-selected lag set: {chosen}")
    return chosen


In [52]:
# Training
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

def train_single_model(X_train, y_train, X_val, y_val, config, verbose=False):
    # Convert to tensors
    X_train_tensor = torch.FloatTensor(X_train).to(config.device)
    y_train_tensor = torch.FloatTensor(y_train).to(config.device)
    X_val_tensor = torch.FloatTensor(X_val).to(config.device)
    y_val_tensor = torch.FloatTensor(y_val).to(config.device)

    # Create data loaders
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True) #drop_last = True)

    # Initialize model
    model_cls = LSTMWithBatchNorm if isinstance(config, Config_LSTM) else MLPWithBatchNorm
    model = model_cls(X_train.shape[1], config.hidden_layers).to(config.device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

    # Early stopping
    early_stopping = EarlyStopping(patience=config.early_stopping_rounds)

    # Track best weights
    best_val   = float('inf')
    best_state = None
    best_epoch = 0

    # Training loop
    train_losses = []
    val_losses = []

    for epoch in range(config.epochs):
        # Training phase
        model.train()
        epoch_train_loss = 0.0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            out  = model(batch_X)
            loss = criterion(out, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) #gradient clipping?
            optimizer.step()
            epoch_train_loss += loss.item()

        avg_train_loss = epoch_train_loss / max(1, len(train_loader))
        train_losses.append(avg_train_loss)

        # # Validation phase
        model.eval()
        with torch.inference_mode():
            vout  = model(X_val_tensor)
            vloss = criterion(vout, y_val_tensor).item()
        val_losses.append(vloss)

        if (best_val - vloss) > early_stopping.min_delta:
            best_val   = vloss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1

        #Early stopping
        early_stopping(vloss)
        if early_stopping.early_stop:
            if verbose:
                print(f"Early stopping at epoch {epoch+1}; best={best_epoch} val={best_val:.6f}")
            break

        if verbose and (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{config.epochs}  Train: {avg_train_loss:.6f}  Val: {vloss:.6f}")

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, train_losses, val_losses

def set_seed_all(seed): # for random ensemble
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def train_ensemble(X_train, y_train, X_val, y_val, config): #better ensemble trainer
    models = []
    base_seed = 42
    print(f"Training ensemble of {config.n_ensembles} models...")
    for i in range(config.n_ensembles):
        print(f"Training model {i+1}/{config.n_ensembles}")
        set_seed_all(base_seed + i)
        model, _, _ = train_single_model(X_train, y_train, X_val, y_val, config, verbose=False)
        models.append(model)
    return models

def predict_ensemble(models, X, config): #make prediction using ensembles
    X_tensor = torch.FloatTensor(X).to(config.device)
    predictions = []

    for model in models:
        model.eval()
        with torch.no_grad():
            predictions.append(model(X_tensor).cpu().numpy())

    # Average predictions
    return np.stack(predictions, axis=0)

In [53]:
# Metrics
def calculate_mse(y_true, y_pred): #MSE
    return mean_squared_error(y_true, y_pred)

def calculate_qlike(y_true, y_pred):
    y_true = np.maximum(y_true, 1e-8) #ensure non zero
    y_pred = np.maximum(y_pred, 1e-8)

    # QLIKE formula
    qlike = np.mean(y_true / y_pred - np.log(y_true / y_pred) - 1)
    return qlike

In [54]:
from pandas import Timedelta

def expanding_window_forecast_pooled(rv_dict, config):
    assert hasattr(config, "train_val_tickers"), "config.train_val_tickers must be set"
    assert hasattr(config, "test_ticker"), "config.test_ticker must be set"

    test_ticker = config.test_ticker
    train_val_tickers = list(config.train_val_tickers)

    # Build lagged data for *all* tickers first
    if config.use_bic:
        # build with superset of lags so we can subset per-window
        all_lags = list(range(1, 45))
    else:
        all_lags = list(config.lag_candidates)

    df_all = create_lagged_features_multi(rv_dict, all_lags)

    # decide which tickers define the pooled "common start"
    # (includes the test ticker; if you want training tickers only, see note below)
    selected_tickers = set(train_val_tickers + [test_ticker])

    # restrict to selected tickers
    df_sel = df_all[df_all['ticker'].isin(selected_tickers)]

    # first usable (post-lag) date per ticker, then take the latest -> common start
    per_ticker_start = df_sel.groupby('ticker').apply(lambda g: g.index.min())
    common_start = per_ticker_start.max()

    # keep test-ticker rows from the common start onward
    df_test = df_all[(df_all['ticker'] == test_ticker) & (df_all.index >= common_start)]

    if df_test.empty:
        raise ValueError(
            f"No data for test ticker '{test_ticker}' on/after common start {common_start.date()}."
        )

    # Dates anchored to the common pooled start (not the test-ticker start)
    start_date = common_start
    initial_train_end = start_date + pd.DateOffset(years=config.initial_train_years)
    current_test_start = initial_train_end + pd.DateOffset(years=config.validation_years)

    # (optional) visibility for debugging:
    print("\n[POOLED] First usable date per ticker (post-lag):")
    print(per_ticker_start.sort_index())
    print(f"[POOLED] Common pooled start: {common_start.date()}")


    print(f"\n[POOLED] Starting expanding-window forecast for test ticker: {test_ticker}")
    print(f"Initial training period (anchored to {test_ticker}): {start_date.date()} to {initial_train_end.date()}")
    print(f"Using log transform: {config.use_log_transform} | Target scaler: {config.target_scaler}")
    print(f"Train/Validation tickers: {sorted(set(train_val_tickers))}")

    # Collect results
    all_results = []

    # If not using BIC, we will reuse a fixed set of feature columns
    if not config.use_bic:
        feature_cols_fixed = [c for c in df_all.columns if c.startswith('lag_')]

    window_count = 0
    while current_test_start < df_test.index[-1]:
        window_count += 1

        # Define date slices relative to the test ticker
        train_end = current_test_start - pd.DateOffset(years=config.validation_years)
        val_start = train_end + Timedelta(days=1)
        val_end   = current_test_start + Timedelta(days=1)
        test_end  = min(current_test_start + pd.DateOffset(months=config.test_months),
                        df_test.index[-1]) + Timedelta(days=2)

        # Slice pooled data by date first
        train_slice = df_all.loc[start_date:train_end].copy()
        val_slice   = df_all.loc[val_start:val_end].copy()
        test_slice  = df_all.loc[val_end + Timedelta(days=1):test_end].copy()

        # Keep only the selected training/validation tickers
        train_slice = train_slice[train_slice['ticker'].isin(train_val_tickers + [test_ticker])]
        val_slice   = val_slice[val_slice['ticker'].isin(train_val_tickers + [test_ticker])]
        # For testing, keep only the test ticker
        test_slice  = test_slice[test_slice['ticker'] == test_ticker]

        if len(test_slice) == 0:
            break

        print(f"\n--- [POOLED] Window {window_count} ---")
        print(f"Train: {start_date.date()} to {train_end.date()}  (rows: {len(train_slice)})")
        print(f"Val:   {val_start.date()} to {val_end.date()}    (rows: {len(val_slice)})")
        print(f"Test:  {(val_end + Timedelta(days=1)).date()} to {test_end.date()} (rows: {len(test_slice)})")

        # Choose lag set
        if config.use_bic:
            # Compute BIC on the *test ticker's* training RV series only
            train_test_only = df_all[(df_all['ticker']==test_ticker) & (df_all.index>=start_date) & (df_all.index<=train_end)]
            bic_lags = calculate_BIC(train_test_only['RV_daily'])
            feature_cols = [f"lag_{k}" for k in bic_lags]
        else:
            feature_cols = feature_cols_fixed

        # Prepare features
        X_train = train_slice[feature_cols].values
        X_val   = val_slice[feature_cols].values
        X_test  = test_slice[feature_cols].values

        # Targets (optionally in log space)
        y_train = train_slice[['RV_daily']].values
        y_val   = val_slice[['RV_daily']].values
        y_test  = test_slice[['RV_daily']].values

        if config.use_log_transform:
            y_train = np.log(np.maximum(y_train, 1e-8))
            y_val   = np.log(np.maximum(y_val,   1e-8))
            y_test  = np.log(np.maximum(y_test,  1e-8))

        # Feature scaling (fit on pooled training set only)
        feature_scaler = StandardScaler()
        X_train_sc = feature_scaler.fit_transform(X_train)
        X_val_sc   = feature_scaler.transform(X_val)
        X_test_sc  = feature_scaler.transform(X_test)

        # Optional target scaling
        if config.target_scaler:
            target_scaler = StandardScaler()
            y_train_sc = target_scaler.fit_transform(y_train)
            y_val_sc   = target_scaler.transform(y_val)
        else:
            target_scaler = None
            y_train_sc, y_val_sc = y_train, y_val

        # Train ensemble
        models = train_ensemble(X_train_sc, y_train_sc, X_val_sc, y_val_sc, config)

        # Predictions in scaled/log space
        train_pred_sc_all = predict_ensemble(models, X_train_sc, config)
        test_pred_sc_all  = predict_ensemble(models, X_test_sc,  config)

        # Inverse target scaling to log space (if any)
        if config.target_scaler and target_scaler is not None:
            train_log_all = np.stack([target_scaler.inverse_transform(p) for p in train_pred_sc_all], axis=0)
            test_log_all  = np.stack([target_scaler.inverse_transform(p)  for p in test_pred_sc_all],  axis=0)
        else:
            train_log_all, test_log_all = train_pred_sc_all, test_pred_sc_all

        # Undo log -> RV space, then arithmetic mean across ensemble
        if config.use_log_transform:
            train_pred = np.mean(np.exp(train_log_all), axis=0)
            test_pred  = np.mean(np.exp(test_log_all),  axis=0)
            y_train_original = np.exp(y_train)
            y_test_original  = np.exp(y_test)
        else:
            train_pred = np.mean(train_log_all, axis=0)
            test_pred  = np.mean(test_log_all,  axis=0)
            y_train_original = y_train
            y_test_original  = y_test

        # Compute per-window metrics
        train_mse   = calculate_mse(y_train_original, train_pred)
        test_mse    = calculate_mse(y_test_original,  test_pred)
        train_qlike = calculate_qlike(y_train_original, train_pred)
        test_qlike  = calculate_qlike(y_test_original,  test_pred)

        print(f"In-sample MSE: {train_mse:.6f}, QLIKE: {train_qlike:.6f}")
        print(f"Out-of-sample MSE: {test_mse:.6f}, QLIKE: {test_qlike:.6f}")

        # Row-wise results for the test ticker
        for i, date in enumerate(test_slice.index):
            all_results.append({
                "ticker": test_ticker,
                "date": date,
                "real_RV": y_test_original[i, 0],
                "predicted_RV": test_pred[i, 0],
                "window": window_count,
                "train_mse": train_mse,
                "test_mse": test_mse,
                "train_qlike": train_qlike,
                "test_qlike": test_qlike
            })

        # Slide one month forward
        current_test_start = current_test_start + pd.DateOffset(months=1)

    return pd.DataFrame(all_results)

In [55]:
# Visualisation
def plot_results(results_df):
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    # Time series of actual vs predicted
    ax1 = axes[0]
    ax1.plot(results_df['date'], results_df['real_RV'], label='Actual RV', alpha=0.7, linewidth=1)
    ax1.plot(results_df['date'], results_df['predicted_RV'], label='Predicted RV', alpha=0.7, linewidth=1)
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Realised Volatility')
    ax1.set_title('Actual vs Predicted Realised Volatility')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Scatter plot
    ax2 = axes[1]
    ax2.scatter(results_df['real_RV'], results_df['predicted_RV'], alpha=0.5, s=10)

    # diagonal line
    min_val = min(results_df['real_RV'].min(), results_df['predicted_RV'].min())
    max_val = max(results_df['real_RV'].max(), results_df['predicted_RV'].max())
    ax2.plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.5, label='Perfect Prediction')

    ax2.set_xlabel('Actual RV')
    ax2.set_ylabel('Predicted RV')
    ax2.set_title('Actual vs Predicted RV (Scatter Plot)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('mlp_rv_forecast_results.png', dpi=300, bbox_inches='tight')
    plt.show()

def print_summary_statistics(results_df): #summary statistics of evaluation metrics
    print("\n" + "="*60)
    print("SUMMARY STATISTICS")
    print("="*60)

    # Overall metrics
    overall_mse = calculate_mse(results_df['real_RV'], results_df['predicted_RV'])
    overall_qlike = calculate_qlike(results_df['real_RV'].values, results_df['predicted_RV'].values)

    print(f"\nOverall Out-of-Sample Performance:")
    print(f"MSE: {overall_mse:.6f}")
    print(f"QLIKE: {overall_qlike:.6f}")

    # Average metrics across windows
    avg_train_mse = results_df.groupby('window')['train_mse'].first().mean()
    avg_test_mse = results_df.groupby('window')['test_mse'].first().mean()
    avg_train_qlike = results_df.groupby('window')['train_qlike'].first().mean()
    avg_test_qlike = results_df.groupby('window')['test_qlike'].first().mean()

    print(f"\nAverage Performance Across Windows:")
    print(f"In-Sample - MSE: {avg_train_mse:.6f}, QLIKE: {avg_train_qlike:.6f}")
    print(f"Out-of-Sample - MSE: {avg_test_mse:.6f}, QLIKE: {avg_test_qlike:.6f}")

    # Correlation
    correlation = results_df['real_RV'].corr(results_df['predicted_RV'])
    print(f"\nCorrelation between Actual and Predicted: {correlation:.4f}")

    # Number of windows
    n_windows = results_df['window'].nunique()
    print(f"\nTotal number of expanding windows: {n_windows}")
    print(f"Total predictions made: {len(results_df)}")

In [ ]:
# MLP
def main_mlp():  # run for MLP forecast
    print("=" * 60)
    print("MLP NEURAL NETWORK")
    print("=" * 60)

    # Initialize configuration
    config = Config_MLP()
    print(f"\nUsing device: {config.device}")

    # --- Define pooled-training settings here ---
    config.train_val_tickers = ["SPY","XLB","XLE","XLF","XLI","XLK","XLP","XLU","XLV","XLY"]
    config.test_ticker = "SPY"
    config.data_dir = "."
    config.do_pooled = True


    tickers_needed = sorted(set(config.train_val_tickers + [config.test_ticker]))
    rv_dict = load_multiple_tickers(tickers_needed, data_dir=config.data_dir)

    # Run pooled expanding-window forecast
    results_df = expanding_window_forecast_pooled(rv_dict, config)

    # Save results
    if config.use_bic:
        output_filename = f"mlp_pooled_bic_rv_forecast_{config.test_ticker}.csv"
    else:
        output_filename = f"mlp_pooled_rv_forecast_{config.test_ticker}.csv"
    results_df.to_csv(output_filename, index=False)
    print(f"\nResults saved to {output_filename}")
    print_summary_statistics(results_df)
    print("\nGenerating plots...")
    plot_results(results_df)
    return

if __name__ == "__main__":
    main_mlp()

MLP NEURAL NETWORK

Using device: cpu
Loading SPY from ./SPY.csv ...
Data loaded: 7301 observations
Date range: 1996-01-02 00:00:00 to 2025-04-29 00:00:00
Loading XLB from ./XLB.csv ...
Data loaded: 6257 observations
Date range: 1999-01-04 00:00:00 to 2025-04-29 00:00:00
Loading XLE from ./XLE.csv ...
Data loaded: 6501 observations
Date range: 1998-12-22 00:00:00 to 2025-04-29 00:00:00
Loading XLF from ./XLF.csv ...
Data loaded: 6512 observations
Date range: 1998-12-22 00:00:00 to 2025-04-29 00:00:00
Loading XLI from ./XLI.csv ...
Data loaded: 6036 observations
Date range: 1999-01-04 00:00:00 to 2025-04-29 00:00:00
Loading XLK from ./XLK.csv ...
Data loaded: 6512 observations
Date range: 1998-12-22 00:00:00 to 2025-04-29 00:00:00
Loading XLP from ./XLP.csv ...
Data loaded: 6486 observations
Date range: 1998-12-22 00:00:00 to 2025-04-29 00:00:00
Loading XLU from ./XLU.csv ...
Data loaded: 6209 observations
Date range: 1998-12-23 00:00:00 to 2025-04-29 00:00:00
Loading XLV from ./XLV.csv

In [ ]:
# LSTM
def main_lstm():  # run for LSTM forecast
    print("=" * 60)
    print("LSTM NEURAL NETWORK")
    print("=" * 60)

    # Initialize configuration
    config = Config_LSTM()
    print(f"\nUsing device: {config.device}")

    # --- Define pooled-training settings here ---
    config.train_val_tickers = ["SPY","XLB","XLE","XLF","XLI","XLK","XLP","XLU","XLV","XLY"]
    config.test_ticker = "SPY"
    config.data_dir = "."
    config.do_pooled = True

        # Load all required tickers
    tickers_needed = sorted(set(config.train_val_tickers + [config.test_ticker]))
    rv_dict = load_multiple_tickers(tickers_needed, data_dir=config.data_dir)

    # Run pooled expanding-window forecast
    results_df = expanding_window_forecast_pooled(rv_dict, config)

    # Save results
    if config.use_bic:
        output_filename = f"lstm_pooled_bic_rv_forecast_{config.test_ticker}.csv"
    else:
        output_filename = f"lstm_pooled_rv_forecast_{config.test_ticker}.csv"
    results_df.to_csv(output_filename, index=False)
    print(f"\nResults saved to {output_filename}")
    print_summary_statistics(results_df)
    print("\nGenerating plots...")
    plot_results(results_df)
    return

if __name__ == "__main__":
    main_lstm()